In [2]:
from pathlib import Path

import h5py
import numpy as np

from md_Helpers import cavitation


# ============================================================
# Select and load/create the cavitation initial state
# ============================================================

cavitation_result = cavitation.get_or_create_cavitation_state(
    n_fcc_cells=30,
    target_rho=0.755,
    kT=0.800,
    source_nsteps=1_000_000,
    radius=2.500,

    source_seed=1,
    source_phase_name="randomization",

    random_location=False,
    bubble_center=None,
    bubble_seed=1,

    overwrite=False,
    overwrite_source=False,

    # Do not launch a missing thermalization run.
    create_source_if_missing=False,
)


# ============================================================
# Print the returned result
# ============================================================

print("\nCAVITATION RESULT")
print("=" * 100)
print("Status:              ", cavitation_result.get("status"))
print("Created new:         ", cavitation_result.get("created_new"))

print("\nPATHS")
for name, value in cavitation_result.get("paths", {}).items():
    print(f"{name:25} {value}")

print("\nCREATION INFO")
creation_info = cavitation_result.get("creation_info", {})

if creation_info:
    for name in sorted(creation_info):
        value = creation_info[name]

        if isinstance(value, np.ndarray):
            print(
                f"{name:25} ndarray, "
                f"shape={value.shape}, dtype={value.dtype}"
            )
        else:
            print(f"{name:25} {value}")
else:
    print("(no creation information returned)")

print("\nSOURCE RESULT")
source_result = cavitation_result.get("source_result", {})
print("Created new:         ", source_result.get("created_new"))

for name, value in source_result.get("paths", {}).items():
    print(f"source_{name:18} {value}")


# ============================================================
# Locate the cavitation creation-metadata file
# ============================================================

metadata_path = Path(
    cavitation_result["paths"]["creation_metadata_path"]
)

print("\nCAVITATION METADATA FILE")
print("=" * 100)
print(metadata_path)

if not metadata_path.exists():
    raise FileNotFoundError(
        "No cavitation metadata file exists. "
        f"Cavitation status was: {cavitation_result.get('status')!r}\n"
        f"Expected file: {metadata_path}"
    )


# ============================================================
# Print every /metadata/... section
# ============================================================

def readable(value):
    """Convert HDF5 and NumPy values into readable values."""
    if isinstance(value, bytes):
        return value.decode("utf-8", errors="replace")

    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, np.ndarray):
        return np.array2string(
            value,
            threshold=np.inf,
            max_line_width=120,
        )

    return value


with h5py.File(metadata_path, "r") as hdf:
    if "metadata" not in hdf:
        raise KeyError(f"No /metadata group exists in {metadata_path}")

    metadata_groups = ["/metadata"]

    def find_groups(name, obj):
        if isinstance(obj, h5py.Group):
            metadata_groups.append(f"/metadata/{name}")

    hdf["metadata"].visititems(find_groups)

    for group_path in metadata_groups:
        group = hdf[group_path]

        print("\n")
        print("=" * 100)
        print(group_path)
        print("=" * 100)

        item_number = 1

        # Attributes saved directly on this group
        if group.attrs:
            print("\nATTRIBUTES")

            for name in sorted(group.attrs):
                value = readable(group.attrs[name])

                print(f"\n[{item_number}] {name}")
                print("    storage: attribute")
                print(f"    type:    {type(value).__name__}")
                print(f"    value:   {value}")

                item_number += 1

        # Datasets saved directly inside this group
        datasets = [
            (name, obj)
            for name, obj in group.items()
            if isinstance(obj, h5py.Dataset)
        ]

        if datasets:
            print("\nDATASETS")

            for name, dataset in sorted(datasets):
                value = readable(dataset[()])

                print(f"\n[{item_number}] {name}")
                print("    storage: dataset")
                print(f"    dtype:   {dataset.dtype}")
                print(f"    shape:   {dataset.shape}")
                print(f"    value:\n{value}")

                item_number += 1

        if not group.attrs and not datasets:
            print("\n(empty container group)")

Thermalized state exists: checking phase separation.
Loaded existing thermalized state:
/exp/e961/data/MDsims-data/pnichols/Thermalized_States_v3/FCC/n_cells_30/rho_0.755/kT_0.800/nsteps_1000000/seed_1/randomization.gsd
Thermalized state good; continuing to cavitation.


Loaded existing cavitation initial state:
/exp/e961/data/MDsims-data/pnichols/Cavitation_States_v3/FCC/n_cells_30/source_rho_0.755/kT_0.800/source_nsteps_1000000/source_seed_1/source_phase_randomization/radius_2.500/center_box/cavitation_initial.gsd

CAVITATION RESULT
Status:               loaded_initial
Created new:          False

PATHS
folder                    /exp/e961/data/MDsims-data/pnichols/Cavitation_States_v3/FCC/n_cells_30/source_rho_0.755/kT_0.800/source_nsteps_1000000/source_seed_1/source_phase_randomization/radius_2.500/center_box
state_path                /exp/e961/data/MDsims-data/pnichols/Cavitation_States_v3/FCC/n_cells_30/source_rho_0.755/kT_0.800/source_nsteps_1000000/source_seed_1/source_phase_ran

In [3]:
# ============================================================
# Clean /metadata/creation in all cavitation initial states
# ============================================================

from collections import Counter
from importlib import reload

from md_Helpers import metadata as metadata_helpers
from md_Helpers.paths import CAVITATION_STATES_V3_ROOT

reload(metadata_helpers)

# Preview first. Change to True after reviewing the report.
APPLY_CHANGES = True
PRINT_EACH_FILE = False

reports = metadata_helpers.cleanup_cavitation_creation_metadata_tree(
    root=CAVITATION_STATES_V3_ROOT,
    dry_run=not APPLY_CHANGES,
)

if not reports:
    raise RuntimeError(
        f"No cavitation_creation.hdf5 files found under "
        f"{CAVITATION_STATES_V3_ROOT}"
    )

status_counts = Counter(report["status"] for report in reports)
errors = [
    report
    for report in reports
    if report["status"] == "error"
]

removed_field_counts = Counter(
    item["path"]
    for report in reports
    for item in report.get("removed", [])
)

print("=" * 100)
print("CAVITATION CREATION-METADATA CLEANUP")
print("=" * 100)
print("Root:          ", CAVITATION_STATES_V3_ROOT)
print("Mode:          ", "APPLY CHANGES" if APPLY_CHANGES else "DRY RUN")
print("Files checked: ", len(reports))

print("\nSTATUS")
for status, count in sorted(status_counts.items()):
    print(f"{status:15} {count}")

print("\nFIELDS")
print("=" * 100)
for field, count in sorted(removed_field_counts.items()):
    action = "removed from" if APPLY_CHANGES else "would remove from"
    print(f"{field:70} {action} {count} files")

if PRINT_EACH_FILE:
    print("\nFILES")
    print("=" * 100)

    for report in reports:
        if report["status"] not in {"would_clean", "cleaned"}:
            continue

        print(
            f"\n{report['status'].upper()}: "
            f"{report['hdf5_path']}"
        )

        for item in report["removed"]:
            print(
                f"  - /{item['path']} "
                f"({item['storage']})"
            )

print("\nERRORS")
print("=" * 100)

if errors:
    for report in errors:
        print(f"\nERROR: {report['hdf5_path']}")
        print(f"       {report['error']}")
else:
    print("No errors found.")

if not APPLY_CHANGES:
    print("\nDRY RUN ONLY: no files were modified.")
    print("Set APPLY_CHANGES = True and rerun to perform the cleanup.")
else:
    print("\nCleanup applied.")

    # Verify that all readable creation files are now clean.
    verification = (
        metadata_helpers.cleanup_cavitation_creation_metadata_tree(
            root=CAVITATION_STATES_V3_ROOT,
            dry_run=True,
        )
    )

    remaining = [
        report
        for report in verification
        if report["status"] == "would_clean"
    ]
    verification_errors = [
        report
        for report in verification
        if report["status"] == "error"
    ]

    print("Files still needing cleanup:", len(remaining))
    print("Files with errors:          ", len(verification_errors))

CAVITATION CREATION-METADATA CLEANUP
Root:           /exp/e961/data/MDsims-data/pnichols/Cavitation_States_v3
Mode:           APPLY CHANGES
Files checked:  760

STATUS
already_clean   760

FIELDS

ERRORS
No errors found.

Cleanup applied.
Files still needing cleanup: 0
Files with errors:           0


In [4]:
# ============================================================
# Remove /metadata/paths from all cavitation initial states
# ============================================================

from collections import Counter
from importlib import reload

from md_Helpers import metadata as metadata_helpers
from md_Helpers.paths import CAVITATION_STATES_V3_ROOT

reload(metadata_helpers)

reports = metadata_helpers.cleanup_cavitation_creation_metadata_tree(
    root=CAVITATION_STATES_V3_ROOT,
    dry_run=False,
)

status_counts = Counter(report["status"] for report in reports)
errors = [
    report
    for report in reports
    if report["status"] == "error"
]
removed_counts = Counter(
    item["path"]
    for report in reports
    for item in report.get("removed", [])
)

print("=" * 100)
print("CAVITATION INITIAL-STATE PATH CLEANUP")
print("=" * 100)
print("Files checked:", len(reports))

print("\nSTATUS")
for status, count in sorted(status_counts.items()):
    print(f"{status:15} {count}")

print("\nREMOVED")
for path, count in sorted(removed_counts.items()):
    print(f"/{path:70} {count} files")

print("\nERRORS")
if errors:
    for report in errors:
        print(f"\nERROR: {report['hdf5_path']}")
        print(f"       {report['error']}")
else:
    print("No errors found.")

# Verify nothing remains to clean.
verification = metadata_helpers.cleanup_cavitation_creation_metadata_tree(
    root=CAVITATION_STATES_V3_ROOT,
    dry_run=True,
)

remaining = [
    report
    for report in verification
    if report["status"] == "would_clean"
]
verification_errors = [
    report
    for report in verification
    if report["status"] == "error"
]

print("\nVERIFICATION")
print("Files still needing cleanup:", len(remaining))
print("Files with errors:          ", len(verification_errors))

CAVITATION INITIAL-STATE PATH CLEANUP
Files checked: 760

STATUS
cleaned         760

REMOVED
/metadata/paths                                                         760 files
/metadata/paths/creation_metadata_path                                  760 files
/metadata/paths/state_path                                              760 files

ERRORS
No errors found.

VERIFICATION
Files still needing cleanup: 0
Files with errors:           0


In [5]:
# ============================================================
# Clean /metadata/source in all cavitation initial states
# ============================================================

from collections import Counter
from importlib import reload

from md_Helpers import metadata as metadata_helpers
from md_Helpers.paths import CAVITATION_STATES_V3_ROOT

reload(metadata_helpers)

reports = metadata_helpers.cleanup_cavitation_creation_metadata_tree(
    root=CAVITATION_STATES_V3_ROOT,
    dry_run=False,
)

status_counts = Counter(report["status"] for report in reports)
errors = [
    report
    for report in reports
    if report["status"] == "error"
]
removed_counts = Counter(
    item["path"]
    for report in reports
    for item in report.get("removed", [])
)

print("=" * 100)
print("CAVITATION INITIAL-STATE SOURCE CLEANUP")
print("=" * 100)
print("Files checked:", len(reports))

print("\nSTATUS")
for status, count in sorted(status_counts.items()):
    print(f"{status:15} {count}")

print("\nREMOVED")
for path, count in sorted(removed_counts.items()):
    print(f"/{path:75} {count} files")

print("\nERRORS")
if errors:
    for report in errors:
        print(f"\nERROR: {report['hdf5_path']}")
        print(f"       {report['error']}")
else:
    print("No errors found.")

# Verify the resulting schema.
verification = metadata_helpers.cleanup_cavitation_creation_metadata_tree(
    root=CAVITATION_STATES_V3_ROOT,
    dry_run=True,
)

remaining = [
    report
    for report in verification
    if report["status"] == "would_clean"
]
verification_errors = [
    report
    for report in verification
    if report["status"] == "error"
]

print("\nVERIFICATION")
print("Files still needing cleanup:", len(remaining))
print("Files with errors:          ", len(verification_errors))

CAVITATION INITIAL-STATE SOURCE CLEANUP
Files checked: 760

STATUS
already_clean   760

REMOVED

ERRORS
No errors found.

VERIFICATION
Files still needing cleanup: 0
Files with errors:           0


In [6]:
# ============================================================
# Clean /metadata/state in all cavitation initial states
# ============================================================

from collections import Counter
from importlib import reload

from md_Helpers import metadata as metadata_helpers
from md_Helpers.paths import CAVITATION_STATES_V3_ROOT

reload(metadata_helpers)

reports = metadata_helpers.cleanup_cavitation_creation_metadata_tree(
    root=CAVITATION_STATES_V3_ROOT,
    dry_run=False,
)

status_counts = Counter(report["status"] for report in reports)
errors = [
    report
    for report in reports
    if report["status"] == "error"
]
removed_counts = Counter(
    item["path"]
    for report in reports
    for item in report.get("removed", [])
)

print("=" * 100)
print("CAVITATION INITIAL-STATE METADATA CLEANUP")
print("=" * 100)
print("Files checked:", len(reports))

print("\nSTATUS")
for status, count in sorted(status_counts.items()):
    print(f"{status:15} {count}")

print("\nREMOVED")
for path, count in sorted(removed_counts.items()):
    print(f"/{path:70} {count} files")

print("\nERRORS")
if errors:
    for report in errors:
        print(f"\nERROR: {report['hdf5_path']}")
        print(f"       {report['error']}")
else:
    print("No errors found.")

# Verify that all files now match the complete initial-state schema.
verification = metadata_helpers.cleanup_cavitation_creation_metadata_tree(
    root=CAVITATION_STATES_V3_ROOT,
    dry_run=True,
)

remaining = [
    report
    for report in verification
    if report["status"] == "would_clean"
]
verification_errors = [
    report
    for report in verification
    if report["status"] == "error"
]

print("\nVERIFICATION")
print("Files still needing cleanup:", len(remaining))
print("Files with errors:          ", len(verification_errors))

CAVITATION INITIAL-STATE METADATA CLEANUP
Files checked: 760

STATUS
cleaned         760

REMOVED
/metadata/source/source_BoxLength                                       760 files
/metadata/source/source_N                                               760 files
/metadata/source/source_actual_rho                                      760 files
/metadata/source/source_data_version                                    760 files
/metadata/source/source_final_timestep                                  760 files
/metadata/source/source_phase_name                                      760 files
/metadata/source/source_state_kind                                      760 files
/metadata/source/source_target_rho                                      760 files
/metadata/source/source_volume                                          760 files
/metadata/state/data_version                                            760 files
/metadata/state/fcc_cell_size                                           760 files
